In [ ]:
"""
Generating reads
- Build a custom dataset 
- Build a simple transformer model
    - Decoder-only (trained on reads)
    - Encoder-decoder model (trained on reads with DNA context)
- Build the training loop
"""
from typing import Any

import random
from dataclasses import dataclass

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F


torch.manual_seed(42)
random.seed(42)
VOCAB = ["<PAD>", "A", "C", "G", "T", "<BOS>", "<EOS>"]


@dataclass
class ModelConfig:
    block_size: int
    n_blocks: int
    n_heads: int
    head_dim: int


@dataclass
class TrainConfig:
    epochs: int


class SequenceReadDataset(Dataset):
    """
    Each sample is:
    - context: the DNA sequence it originates from
    - reads: the reads that originate from a specific DNA sequence

    so something like: ("ACGAGCATCAGC", ["ACG", "CATC"])

    In normal transformer land, we still would have one read = one sample, i.e.:
    if read = ACG -> input: <S>ACG, output/labels: ACG<E>

    For now, our goal is to build the reads dataset only and later add the context component
    """

    def __init__(self, num_samples) -> None:
        super().__init__()
        self.vocab = VOCAB
        self.encoding = {token: idx for idx, token in enumerate(self.vocab)}
        self.inputs, self.labels = self._make_samples(num_samples)

    def _make_samples(self, num_samples: int):
        all_inputs, all_labels = [], []
        for _ in range(num_samples):
            max_len = 160
            min_len = max_len - 30
            read_len = torch.randint(min_len, max_len, (1,)).item()

            bos_id = self.encoding["<BOS>"]
            eos_id = self.encoding["<EOS>"]
            pad_id = self.encoding["<PAD>"]

            nucleotides = torch.randint(1, 5, size=(read_len,))

            inputs = torch.full(size=(max_len,), fill_value=pad_id, dtype=torch.long)
            inputs[0] = bos_id
            inputs[1 : read_len + 1] = nucleotides

            labels = torch.full(size=(max_len,), fill_value=pad_id, dtype=torch.long)
            labels[:read_len] = nucleotides
            labels[read_len] = eos_id

            all_inputs.append(inputs)
            all_labels.append(labels)

        return all_inputs, all_labels

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.inputs[idx], self.labels[idx]

    def __len__(self):
        return len(self.inputs)


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, head_dim, n_heads, block_size) -> None:
        super().__init__()
        self.emb_dim = head_dim * n_heads
        self.head_dim = head_dim
        self.n_heads = n_heads
        self.w_qkv = nn.Linear(self.emb_dim, self.emb_dim * 3)
        self.w_out = nn.Linear(self.emb_dim, self.emb_dim)

        mask = torch.full((block_size, block_size), -torch.inf).triu(1)
        self.register_buffer("mask", mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        wqv = self.w_qkv(x)
        q, k, v = torch.chunk(wqv, 3, dim=-1)

        # reshape to be multihead attention
        q = q.unflatten(-1, [self.n_heads, self.head_dim]).transpose(1, 2)
        k = k.unflatten(-1, [self.n_heads, self.head_dim]).transpose(1, 2)
        v = v.unflatten(-1, [self.n_heads, self.head_dim]).transpose(1, 2)

        attn = F.scaled_dot_product_attention(q, k, v, self.mask)
        attn = attn.transpose(1, 2).flatten(-2)

        out = self.w_out(attn)

        return out


class CrossAttentionBlock(nn.Module):
    """
    Module that lives
    """

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        return  # TODO implement


class EncoderBlock(nn.Module):
    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x


class DNAEncoder(nn.Module):
    """
    Takes a DNA sequence of length 4096bp and encodes it.
    Keys and values of this module are input to every decoder
    transformer block.
    """

    def __init__(self, block_size: int, n_blocks: int, emb_dim: int) -> None:
        super().__init__()
        self.block_size = block_size
        self.embedding = nn.Embedding(num_embeddings=len(VOCAB), embedding_dim=emb_dim)
        self.pos_embedding = nn.Embedding(
            num_embeddings=block_size, embedding_dim=20
        )  # TODO replace with RoPE
        self.blocks = nn.Sequential(*[EncoderBlock() for _ in range(n_blocks)])
        self.ln_final = nn.LayerNorm(emb_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        positions = torch.arange(0, self.block_size)
        x = self.embedding(x) + self.pos_embedding(positions)
        x = self.blocks(x)
        x = self.ln_final(x)

        return x


class TransformerBlock(nn.Module):
    def __init__(self, head_dim, n_heads, block_size) -> None:
        super().__init__()
        emb_dim = head_dim * n_heads
        self.ln1 = nn.LayerNorm(emb_dim)
        self.self_attn = CausalMultiHeadAttention(head_dim, n_heads, block_size)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.1)
        self.cross_attn = CrossAttentionBlock()
        self.ffn = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.GELU(),  # TODO: replace with SwiGLU?
            nn.Linear(4 * emb_dim, emb_dim),
        )

    def forward(
        self, x: torch.Tensor, dna_context: tuple[torch.Tensor, torch.Tensor]
    ) -> torch.Tensor:
        x = x + self.dropout1(self.self_attn(self.ln1(x)))
        k, v = dna_context
        x = self.cross_attn(q, k, v)

        x = x + self.dropout2(self.ffn(self.ln2(x)))
        return x


class Coaster(nn.Module):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        emb_dim = config.head_dim * config.n_heads
        self.token_embedding = nn.Embedding(num_embeddings=len(VOCAB), embedding_dim=emb_dim)
        self.pos_embedding = nn.Embedding(
            num_embeddings=config.block_size, embedding_dim=emb_dim
        )  # TODO replace with RoPE?
        self.dna_encoder = DNAEncoder(emb_dim)
        self.blocks = nn.ModuleList(
            [
                TransformerBlock(config.head_dim, config.n_heads, config.block_size)
                for _ in range(config.n_blocks)
            ]
        )
        self.lm_head = nn.Linear(emb_dim, len(VOCAB))

    def forward(self, read: torch.Tensor, dna: torch.Tensor) -> torch.Tensor:
        T = read.shape[1]
        pos = torch.arange(T, device=read.device)
        dna_context = self.dna_encoder()
        x = self.token_embedding(read) + self.pos_embedding(pos)
        for block in self.blocks:
            x = block(x, dna_context)
        x = self.lm_head(x)
        return x


train_config = TrainConfig(epochs=100)
model_config = ModelConfig(block_size=160, n_blocks=2, n_heads=4, head_dim=16)
model = Coaster(model_config)

dataset = SequenceReadDataset(num_samples=1)
train_loader = DataLoader(dataset, batch_size=1, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters())

print(model)

for epoch in range(train_config.epochs):
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        out = model(inputs)
        loss = F.cross_entropy(out.transpose(1, 2), labels, ignore_index=0)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1} | Loss: {loss:.4f}")
    break

Coaster(
  (token_embedding): Embedding(7, 64)
  (pos_embedding): Embedding(160, 64)
  (blocks): Sequential(
    (0): TransformerBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attn): CausalMultiHeadAttention(
        (w_qkv): Linear(in_features=64, out_features=192, bias=True)
        (w_out): Linear(in_features=64, out_features=64, bias=True)
      )
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplace=False)
      (ffn): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=256, out_features=64, bias=True)
      )
    )
    (1): TransformerBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attn): CausalMultiHeadAttention(
        (w_qkv): Linear(in_features=64, out_features=192, bias=True)
        (w_out): Linear(in_features=64, ou

In [3]:
import pandas as pd

df = pd.read_parquet("/home/tds122/coaster/data/samples_yeast.parquet")

df

,chr,strand,start_seq,end_seq,start_coverage,end_coverage,input_sequence,fold
0,NC_001133.9,+,1,4000,1,3000,CCACACCACACCCACACACCCACACACCACACCACACACCACACCA...,train
1,NC_001133.9,+,2001,7000,3001,6000,GGGCGGCTTGGAACATGTAGTATTGGGCTAAGTGAGCTCTGATATC...,train
2,NC_001133.9,+,5001,10000,6001,9000,ATTTATCTTTTATAAAAAACCCTTGTTCTACTGACAGGATGGAATA...,train
3,NC_001133.9,+,8001,13000,9001,12000,CTAATTTCATCATCAGTTAAGAAAATAGAATAACAGTTGTATGGGT...,train
4,NC_001133.9,+,10389,15388,11389,14388,CATGTATATAGAAAATCACAGTACAAAAATTTTGAATTTATGTATA...,train
...,...,...,...,...,...,...,...,...
8832,NC_001148.4,-,12904,17903,13904,16903,TTGTAATTATTCCTGCTTAAAATGAATAAAGAAAGTACAACACAAT...,train
8833,NC_001148.4,-,9918,14917,10918,13917,AGTCATGCCAGCAAGATTTAAGATTTTCTTTATTTAGTGTTTATAT...,train
8834,NC_001148.4,-,6918,11917,7918,10917,TAAGAGATATTAGCGCTCTTTTTGGAGTCATTGCAAGGCCAGTTTT...,train
8835,NC_001148.4,-,3918,8917,4918,7917,TTAGAATAACGTAATCGTATCAACGAATCCACTAGGCGCGCGTAAA...,train


In [ ]:
import pandas as pd

df = pd.read_pickle("/home/tds122/coaster/data/samples.pkl")

df